# Hidden State Visualisation — CompLearn Workshop

PCA and UMAP projections of `worker_hidden` across cells and checkpoints.
Flip `USE_FAKE_DATA = False` when real dumps are available.

In [ ]:
# Cell 1 — Imports and config
import sys
import os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

try:
    import umap
except ImportError:
    raise ImportError(
        "umap-learn is not installed. Run:\n"
        "    pip install umap-learn\n"
        "(Note: the package name is 'umap-learn', not 'umap')"
    )

# Ensure repo root is on sys.path so relative imports work when notebook
# is opened from analysis/ or from the repo root.
# __file__ is not defined in notebooks; resolve from CWD heuristic instead.
REPO_ROOT = Path(os.getcwd())
if REPO_ROOT.name == "analysis":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# ── single switch ──────────────────────────────────────────────────────────
USE_FAKE_DATA   = True   # flip to False when real dumps exist
# ──────────────────────────────────────────────────────────────────────────

CELLS           = ["A", "B", "C", "D", "E"]
CHECKPOINTS     = ["early", "mid", "late"]
HIDDEN_SIZE     = 512
N_EXAMPLES      = 200
N_PUZZLES       = 30
DUMPS_DIR       = REPO_ROOT / "dumps"
FIGURES_DIR     = REPO_ROOT / "analysis" / "figures"
RANDOM_SEED     = 42

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"DUMPS_DIR   : {DUMPS_DIR}")
print(f"FIGURES_DIR : {FIGURES_DIR}")
print(f"USE_FAKE_DATA: {USE_FAKE_DATA}")

In [ ]:
# Cell 2 — Fake data generator

# Solve-rate schedule per (cell, checkpoint)
_SOLVE_RATES = {
    "A": {"early": 0.10, "mid": 0.40, "late": 0.70},
    "B": {"early": 0.15, "mid": 0.15, "late": 0.15},  # baseline, flat
    "C": {"early": 0.10, "mid": 0.40, "late": 0.70},
    "D": {"early": 0.10, "mid": 0.40, "late": 0.70},
    "E": {"early": 0.20, "mid": 0.20, "late": 0.20},  # random directions, flat
}

# Cluster std per checkpoint — tightens as training progresses
_CLUSTER_STD = {"early": 2.0, "mid": 1.0, "late": 0.4}


def generate_fake_dumps() -> None:
    """Generate 15 fake dump files (5 cells x 3 checkpoints) into DUMPS_DIR."""
    rng = np.random.default_rng(RANDOM_SEED)

    # Stable per-puzzle cluster centres in hidden-state space — same for all cells
    # and checkpoints (only std changes).  Shape: [N_PUZZLES, HIDDEN_SIZE].
    puzzle_centers_worker = rng.standard_normal((N_PUZZLES, HIDDEN_SIZE)).astype(np.float32) * 4.0

    # Separate subgoal centre directions (orthogonal-ish by construction of random init)
    puzzle_centers_subgoal = rng.standard_normal((N_PUZZLES, HIDDEN_SIZE)).astype(np.float32) * 4.0

    # Assign each of the N_EXAMPLES examples to a puzzle — multiple examples share IDs
    puzzle_ids = rng.integers(0, N_PUZZLES, size=N_EXAMPLES)

    for cell in CELLS:
        for ckpt in CHECKPOINTS:
            out_path = DUMPS_DIR / f"cell_{cell}" / f"{ckpt}.pt"
            out_path.parent.mkdir(parents=True, exist_ok=True)

            std = _CLUSTER_STD[ckpt]

            # worker_hidden: Gaussian around puzzle-specific centre
            worker_hidden = (
                puzzle_centers_worker[puzzle_ids]
                + rng.standard_normal((N_EXAMPLES, HIDDEN_SIZE)).astype(np.float32) * std
            )

            # subgoal_goal
            if cell == "E":
                # Random directions — unit-norm random vectors (no puzzle structure)
                raw = rng.standard_normal((N_EXAMPLES, HIDDEN_SIZE)).astype(np.float32)
                norms = np.linalg.norm(raw, axis=1, keepdims=True)
                subgoal_goal = raw / np.maximum(norms, 1e-8)
            else:
                raw = (
                    puzzle_centers_subgoal[puzzle_ids]
                    + rng.standard_normal((N_EXAMPLES, HIDDEN_SIZE)).astype(np.float32) * std
                )
                norms = np.linalg.norm(raw, axis=1, keepdims=True)
                subgoal_goal = raw / np.maximum(norms, 1e-8)

            # solved: Bernoulli with schedule
            rate = _SOLVE_RATES[cell][ckpt]
            solved = torch.from_numpy(
                rng.random(N_EXAMPLES).astype(np.float32) < rate
            )  # bool tensor

            payload = {
                "worker_hidden":     torch.from_numpy(worker_hidden),   # [200, 512]
                "subgoal_goal":      torch.from_numpy(subgoal_goal),    # [200, 512]
                "solved":            solved,                            # [200] bool
                "puzzle_identifier": torch.from_numpy(puzzle_ids.astype(np.int64)),  # [200]
                "example_index":     torch.arange(N_EXAMPLES, dtype=torch.long),     # [200]
                "metadata": {
                    "checkpoint_path":   f"checkpoints/fake/cell_{cell}/{ckpt}",
                    "config_name":       f"cfg_pretrain_cell{cell}",
                    "hydra_overrides":   [],
                    "model_param_count": 279299,
                    "eval_data_path":    "data/fake_eval.jsonl",
                    "subset_size":       N_EXAMPLES,
                },
            }
            torch.save(payload, out_path)

    print(f"Generated {len(CELLS) * len(CHECKPOINTS)} fake dump files under {DUMPS_DIR}/")


if USE_FAKE_DATA:
    generate_fake_dumps()

In [ ]:
# Cell 3 — Loader

def load_dumps() -> dict:
    """Return dumps[cell][checkpoint] = dict_with_tensors.

    Works identically for fake and real modes — both are .pt files on disk
    at DUMPS_DIR/cell_<X>/<checkpoint>.pt.
    """
    result = {}
    for cell in CELLS:
        result[cell] = {}
        for ckpt in CHECKPOINTS:
            path = DUMPS_DIR / f"cell_{cell}" / f"{ckpt}.pt"
            if not path.exists():
                raise FileNotFoundError(
                    f"Dump file not found: {path}\n"
                    f"  If USE_FAKE_DATA=True, ensure generate_fake_dumps() ran successfully.\n"
                    f"  If USE_FAKE_DATA=False, run scripts/dump_hidden_states.py to produce "
                    f"real dumps in {DUMPS_DIR}/"
                )
            result[cell][ckpt] = torch.load(path, map_location="cpu", weights_only=False)
    print(f"Loaded {len(CELLS) * len(CHECKPOINTS)} dump files.")
    return result


dumps = load_dumps()

# Quick schema check
sample = dumps["A"]["late"]
print("Keys :", list(sample.keys()))
print("worker_hidden shape:", sample["worker_hidden"].shape)
print("subgoal_goal  shape:", sample["subgoal_goal"].shape)
print("solved        shape:", sample["solved"].shape, "dtype:", sample["solved"].dtype)
print("puzzle_identifier:",   sample["puzzle_identifier"].shape)

In [ ]:
# Cell 4 — PCA fitter (shared across checkpoints, fit on late only)

def fit_shared_pca(cell_dumps: dict, n_components: int = 2):
    """Fit PCA on the late checkpoint; project all checkpoints into that space.

    Fitting on `late` only means early/mid are projected into the final
    representation space — trajectory is interpretable as "where did these
    examples start relative to where they ended up".
    """
    final_hidden = cell_dumps["late"]["worker_hidden"].numpy()  # [200, hidden_size]
    pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
    pca.fit(final_hidden)
    projected = {}
    for ckpt in CHECKPOINTS:
        projected[ckpt] = pca.transform(cell_dumps[ckpt]["worker_hidden"].numpy())
    return projected, pca


pca_projections = {}   # pca_projections[cell] = {ckpt: ndarray [200, 2]}
pca_fitters     = {}   # pca_fitters[cell]     = fitted PCA object

for cell in CELLS:
    pca_projections[cell], pca_fitters[cell] = fit_shared_pca(dumps[cell])

print("PCA fitting done. Projection shapes:")
for cell in CELLS:
    shapes = {ckpt: pca_projections[cell][ckpt].shape for ckpt in CHECKPOINTS}
    print(f"  Cell {cell}: {shapes}")

In [ ]:
# Cell 5 — UMAP fitter (same design: fit on late, transform early/mid)
#
# UMAP.transform() is approximate — it uses the learned manifold from fit()
# to place new points, which can show edge effects for early-checkpoint
# representations that lie off the late manifold.  This is acceptable here:
# the visual goal is to show *trajectory* in the final coordinate system,
# and the approximation is consistent across examples.

def fit_shared_umap(cell_dumps: dict, n_components: int = 2):
    """Fit UMAP on the late checkpoint; project all checkpoints into that space."""
    final_hidden = cell_dumps["late"]["worker_hidden"].numpy()
    reducer = umap.UMAP(n_components=n_components, random_state=RANDOM_SEED)
    reducer.fit(final_hidden)
    projected = {}
    for ckpt in CHECKPOINTS:
        projected[ckpt] = reducer.transform(cell_dumps[ckpt]["worker_hidden"].numpy())
    return projected, reducer


umap_projections = {}  # umap_projections[cell] = {ckpt: ndarray [200, 2]}
umap_fitters     = {}

print("Fitting UMAP (this takes ~10–30 s on CPU)…")
for cell in CELLS:
    umap_projections[cell], umap_fitters[cell] = fit_shared_umap(dumps[cell])
    print(f"  Cell {cell} done.")

print("UMAP fitting done.")

In [ ]:
# Cell 6 — Plotting: per-cell 2×3 figures, two coloring versions each

CMAP_PUZZLE = "tab20"
COLOR_SOLVED   = "#4caf50"   # green
COLOR_UNSOLVED = "#9e9e9e"   # gray
POINT_SIZE  = 15
POINT_ALPHA = 0.7


def _scatter(ax, xy, colors, cmap=None, vmin=None, vmax=None, label=None):
    sc = ax.scatter(
        xy[:, 0], xy[:, 1],
        c=colors, cmap=cmap, vmin=vmin, vmax=vmax,
        s=POINT_SIZE, alpha=POINT_ALPHA, linewidths=0,
    )
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xticks([])
    ax.set_yticks([])
    return sc


generated_files = []

for cell in CELLS:
    puzzle_ids_np = dumps[cell]["late"]["puzzle_identifier"].numpy()  # [200]
    solved_np     = dumps[cell]["late"]["solved"].numpy().astype(bool) # [200] bool

    for coloring in ["puzzle", "solved"]:
        fig, axes = plt.subplots(2, 3, figsize=(12, 7))

        methods = ["PCA", "UMAP"]
        proj_dict = {"PCA": pca_projections[cell], "UMAP": umap_projections[cell]}

        for row, method in enumerate(methods):
            for col, ckpt in enumerate(CHECKPOINTS):
                ax = axes[row][col]
                xy = proj_dict[method][ckpt]  # [200, 2]

                if coloring == "puzzle":
                    # Use puzzle_identifier from that checkpoint's dump for coloring
                    pids = dumps[cell][ckpt]["puzzle_identifier"].numpy()
                    _scatter(ax, xy, pids,
                             cmap=CMAP_PUZZLE, vmin=0, vmax=N_PUZZLES - 1)
                else:
                    sol = dumps[cell][ckpt]["solved"].numpy().astype(bool)
                    colors = np.where(sol, COLOR_SOLVED, COLOR_UNSOLVED)
                    _scatter(ax, xy, colors)

                ax.set_title(f"{method} | {ckpt}", fontsize=10)

        color_label = "puzzle" if coloring == "puzzle" else "solved"
        fig.suptitle(f"Cell {cell} — colored by {color_label}", fontsize=13, y=1.01)
        fig.tight_layout()

        fname = FIGURES_DIR / f"cell_{cell}_by_{color_label}.pdf"
        fig.savefig(fname, bbox_inches="tight", dpi=150)
        plt.close(fig)
        generated_files.append(fname)

print(f"Saved {len(generated_files)} figures:")
for f in generated_files:
    print(f"  {f.relative_to(REPO_ROOT)}")

In [ ]:
# Cell 7 — Sanity diagnostics


print("\n── PCA explained variance (late fit) ──────────────────────────────")
print(f"{'Cell':>6}  {'PC1 var':>10}  {'PC2 var':>10}  {'PC1+PC2':>10}")
print("-" * 44)
for cell in CELLS:
    ev = pca_fitters[cell].explained_variance_ratio_
    print(f"  {cell:>4}  {ev[0]:>10.4f}  {ev[1]:>10.4f}  {ev[0]+ev[1]:>10.4f}")

print("\n── Mean L2 distance in PCA space: early → late ─────────────────────")
print(f"{'Cell':>6}  {'mean L2':>12}  {'note':}")
print("-" * 48)
for cell in CELLS:
    early_proj = pca_projections[cell]["early"]  # [200, 2]
    late_proj  = pca_projections[cell]["late"]
    dists = np.linalg.norm(late_proj - early_proj, axis=1)  # [200]
    mean_dist = dists.mean()
    note = "(random directions — expect smaller drift)" if cell == "E" else ""
    print(f"  {cell:>4}  {mean_dist:>12.4f}  {note}")

# Explicit check: cell E drift should be smaller than cell A
dist_A = np.linalg.norm(
    pca_projections["A"]["late"] - pca_projections["A"]["early"], axis=1
).mean()
dist_E = np.linalg.norm(
    pca_projections["E"]["late"] - pca_projections["E"]["early"], axis=1
).mean()
status = "PASS" if dist_E < dist_A else "FAIL"
print(f"\nSanity check — E drift < A drift: {status}  (E={dist_E:.4f}, A={dist_A:.4f})")

In [ ]:
# Cell 8 — Comparison figure: 5 cells × PCA-by-puzzle at late checkpoint
#
# Shared axis limits across all subplots so spatial scale is comparable.

fig, axes = plt.subplots(1, len(CELLS), figsize=(4 * len(CELLS), 4))

# Compute global axis bounds from all late PCA projections
all_xy = np.concatenate(
    [pca_projections[cell]["late"] for cell in CELLS], axis=0
)
margin = 0.5
xlim = (all_xy[:, 0].min() - margin, all_xy[:, 0].max() + margin)
ylim = (all_xy[:, 1].min() - margin, all_xy[:, 1].max() + margin)

for ax, cell in zip(axes, CELLS):
    xy  = pca_projections[cell]["late"]
    pids = dumps[cell]["late"]["puzzle_identifier"].numpy()
    _scatter(ax, xy, pids, cmap=CMAP_PUZZLE, vmin=0, vmax=N_PUZZLES - 1)
    # Use adjustable="box" so explicit xlim/ylim are honoured; matplotlib
    # shrinks the subplot box rather than expanding the data range.
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_title(f"Cell {cell}", fontsize=11)

fig.suptitle("Late checkpoint PCA — colored by puzzle (all cells)", fontsize=12, y=1.02)
fig.tight_layout()

out_path = FIGURES_DIR / "comparison_late_pca_by_puzzle.pdf"
fig.savefig(out_path, bbox_inches="tight", dpi=150)
plt.close(fig)
print(f"Saved: {out_path.relative_to(REPO_ROOT)}")